# Pandas: A Refresher Lesson

Pandas is the main library for working with tabular data, meaning data in rows and columns, like a spreadsheet or a database table. It's built on NumPy arrays.

This lesson goes in order:

1. Series and DataFrame basics
2. Loading data
3. Inspecting data
4. Selecting columns and rows (`[]`, `loc`, `iloc`)
5. Filtering
6. Creating and changing columns
7. Missing values
8. Sorting and ranking
9. Grouping and aggregating (`groupby`)
10. Combining tables (`merge` and `concat`)
11. Dates
12. Chaining operations
13. Practice exercises, with solutions

We'll use a small, made-up dataset of health insurance claims, so the code runs without any downloads. Run the cells in order.

## 1. Setup and core objects

Pandas has two main objects:

- **`Series`**: one column of values, with an index (row labels).
- **`DataFrame`**: a table made of several Series that share the same index.

In [2]:
import pandas as pd
import numpy as np

print(pd.__version__)

s = pd.Series([120, 45, 890], name="claim_cost")
print(s)

3.0.6
0    120
1     45
2    890
Name: claim_cost, dtype: int64


A DataFrame is a dictionary of columns, or a list of row dictionaries:

In [3]:
df = pd.DataFrame({
    "name": ["Ana", "Ben", "Chloe"],
    "age": [34, 58, 41],
    "claim_cost": [120.0, 890.0, 45.0],
})
df

,name,age,claim_cost
0,Ana,34,120.0
1,Ben,58,890.0
2,Chloe,41,45.0


In a notebook, the last line of a cell is displayed automatically. Use `print(df)` if you want the same output in a regular Python script.

## 2. Loading data

Real projects usually load data from files with functions like `pd.read_csv`, `pd.read_excel`, and `pd.read_parquet`. For this lesson, we read CSV text from memory, so the example runs as-is:

In [4]:
from io import StringIO

csv_text = """claim_id,patient_id,date,provider,type,amount,paid
1,P001,2025-01-04,Clinic A,outpatient,120.50,True
2,P002,2025-01-09,Hospital B,inpatient,8450.00,True
3,P001,2025-02-11,Clinic A,outpatient,45.00,False
4,P003,2025-02-20,Clinic C,pharmacy,15.25,True
5,P002,2025-03-02,Hospital B,outpatient,310.00,
6,P004,2025-03-15,Clinic A,outpatient,,True
7,P003,2025-03-28,Hospital B,inpatient,2200.75,True
8,P005,2025-04-01,Clinic C,pharmacy,980.00,True
"""

claims = pd.read_csv(StringIO(csv_text))
claims

,claim_id,patient_id,date,provider,type,amount,paid
0,1,P001,2025-01-04,Clinic A,outpatient,120.50,True
1,2,P002,2025-01-09,Hospital B,inpatient,8450.00,True
2,3,P001,2025-02-11,Clinic A,outpatient,45.00,False
3,4,P003,2025-02-20,Clinic C,pharmacy,15.25,True
4,5,P002,2025-03-02,Hospital B,outpatient,310.00,NaN
5,6,P004,2025-03-15,Clinic A,outpatient,NaN,True
6,7,P003,2025-03-28,Hospital B,inpatient,2200.75,True
7,8,P005,2025-04-01,Clinic C,pharmacy,980.00,True


Notice that `paid` has a missing value, and `amount` has one too. Pandas marks those automatically as `NaN` (not a number) or `<NA>`. We'll handle them later.

To load a real file, you'd write `pd.read_csv("data/claims.csv")`.

Saving a DataFrame works the same way: `claims.to_csv("out.csv", index=False)`.

## 3. Inspecting data

Before you analyze anything, look at the data. These are the usual first checks:

In [5]:
print("shape:", claims.shape)          # (rows, columns)
print()
print(claims.dtypes)
print()
print(claims.head(3))                  # first 3 rows
print()
print(claims.describe())               # summary statistics for numeric columns

shape: (8, 7)

claim_id        int64
patient_id        str
date              str
provider          str
type              str
amount        float64
paid           object
dtype: object

   claim_id patient_id        date    provider        type  amount   paid
0         1       P001  2025-01-04    Clinic A  outpatient   120.5   True
1         2       P002  2025-01-09  Hospital B   inpatient  8450.0   True
2         3       P001  2025-02-11    Clinic A  outpatient    45.0  False

       claim_id       amount
count   8.00000     7.000000
mean    4.50000  1731.642857
std     2.44949  3064.010702
min     1.00000    15.250000
25%     2.75000    82.750000
50%     4.50000   310.000000
75%     6.25000  1590.375000
max     8.00000  8450.000000


`dtypes` shows what pandas thinks each column is. If a number column shows as `object` or the wrong type, check the data before you go further.

`info()` gives a quick overview, including how many values are missing:

In [6]:
claims.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   claim_id    8 non-null      int64  
 1   patient_id  8 non-null      str    
 2   date        8 non-null      str    
 3   provider    8 non-null      str    
 4   type        8 non-null      str    
 5   amount      7 non-null      float64
 6   paid        7 non-null      object 
dtypes: float64(1), int64(1), object(1), str(4)
memory usage: 580.0+ bytes


## 4. Selecting columns and rows

Getting a column returns a Series:

In [7]:
claims["amount"]          # one column -> Series

0     120.50
1    8450.00
2      45.00
3      15.25
4     310.00
5        NaN
6    2200.75
7     980.00
Name: amount, dtype: float64

Getting several columns returns a DataFrame. Note the double brackets:

In [8]:
claims[["patient_id", "amount"]]

,patient_id,amount
0,P001,120.50
1,P002,8450.00
2,P001,45.00
3,P003,15.25
4,P002,310.00
5,P004,NaN
6,P003,2200.75
7,P005,980.00


For rows, use **`loc`** (by label) and **`iloc`** (by position). The rule to remember: `loc` is for names and labels, and `iloc` is for numbers.

In [9]:
print(claims.iloc[0])                     # first row, by position
print()
print(claims.loc[2, ["provider", "amount"]])  # row labeled 2, two columns
print()
print(claims.iloc[1:4, 2:5])             # rows 1-3, columns 2-4 (stop is excluded)

claim_id               1
patient_id          P001
date          2025-01-04
provider        Clinic A
type          outpatient
amount             120.5
paid                True
Name: 0, dtype: object

provider    Clinic A
amount          45.0
Name: 2, dtype: object

         date    provider        type
1  2025-01-09  Hospital B   inpatient
2  2025-02-11    Clinic A  outpatient
3  2025-02-20    Clinic C    pharmacy


With the default index, `loc` and `iloc` often look the same, but they behave differently once you set a custom index. Keep that in mind.

## 5. Filtering

Filters work like NumPy masks. Combine conditions with `&` and `|`, and wrap each one in parentheses:

In [10]:
large = claims[claims["amount"] > 500]
large

,claim_id,patient_id,date,provider,type,amount,paid
1,2,P002,2025-01-09,Hospital B,inpatient,8450.00,True
6,7,P003,2025-03-28,Hospital B,inpatient,2200.75,True
7,8,P005,2025-04-01,Clinic C,pharmacy,980.00,True


In [11]:
clinic_outpatient = claims[
    (claims["provider"] == "Clinic A") & (claims["type"] == "outpatient")
]
clinic_outpatient

,claim_id,patient_id,date,provider,type,amount,paid
0,1,P001,2025-01-04,Clinic A,outpatient,120.5,True
2,3,P001,2025-02-11,Clinic A,outpatient,45.0,False
5,6,P004,2025-03-15,Clinic A,outpatient,NaN,True


Other useful filter methods:

In [12]:
print(claims[claims["type"].isin(["inpatient", "pharmacy"])])
print()
print(claims[claims["provider"].str.contains("Hospital")])

   claim_id patient_id        date    provider       type   amount  paid
1         2       P002  2025-01-09  Hospital B  inpatient  8450.00  True
3         4       P003  2025-02-20    Clinic C   pharmacy    15.25  True
6         7       P003  2025-03-28  Hospital B  inpatient  2200.75  True
7         8       P005  2025-04-01    Clinic C   pharmacy   980.00  True

   claim_id patient_id        date    provider        type   amount  paid
1         2       P002  2025-01-09  Hospital B   inpatient  8450.00  True
4         5       P002  2025-03-02  Hospital B  outpatient   310.00   NaN
6         7       P003  2025-03-28  Hospital B   inpatient  2200.75  True


## 6. Creating and changing columns

Assigning to a new column name creates it. The operation works on the whole column at once, with no loop:

In [13]:
claims["amount_with_tax"] = claims["amount"] * 1.07
claims[["amount", "amount_with_tax"]]

,amount,amount_with_tax
0,120.50,128.9350
1,8450.00,9041.5000
2,45.00,48.1500
3,15.25,16.3175
4,310.00,331.7000
5,NaN,NaN
6,2200.75,2354.8025
7,980.00,1048.6000


Use `np.where` for conditional values, or `.loc` to set values under a condition:

In [14]:
claims["size"] = np.where(claims["amount"] > 500, "large", "small")

claims.loc[claims["paid"] == False, "size"] = "unpaid"

claims[["amount", "paid", "size"]]

,amount,paid,size
0,120.50,True,small
1,8450.00,True,large
2,45.00,False,unpaid
3,15.25,True,small
4,310.00,NaN,small
5,NaN,True,small
6,2200.75,True,large
7,980.00,True,large


Note the `== False` comparison above. A missing value in `paid` will not match either test, so it keeps its previous value. We'll handle that properly in the next section.

To remove a column:

In [15]:
claims = claims.drop(columns=["amount_with_tax"])
claims.columns.tolist()

['claim_id',
 'patient_id',
 'date',
 'provider',
 'type',
 'amount',
 'paid',
 'size']

## 7. Missing values

Missing data shows up in real datasets all the time. Pandas gives you three main tools: find them, drop them, or fill them.

In [16]:
print(claims.isna())                   # True where a value is missing
print()
print(claims.isna().sum())             # count of missing values per column

   claim_id  patient_id   date  provider   type  amount   paid   size
0     False       False  False     False  False   False  False  False
1     False       False  False     False  False   False  False  False
2     False       False  False     False  False   False  False  False
3     False       False  False     False  False   False  False  False
4     False       False  False     False  False   False   True  False
5     False       False  False     False  False    True  False  False
6     False       False  False     False  False   False  False  False
7     False       False  False     False  False   False  False  False

claim_id      0
patient_id    0
date          0
provider      0
type          0
amount        1
paid          1
size          0
dtype: int64


In [17]:
print("rows with any missing value:")
print(claims[claims.isna().any(axis=1)])

rows with any missing value:
   claim_id patient_id        date    provider        type  amount  paid  \
4         5       P002  2025-03-02  Hospital B  outpatient   310.0   NaN   
5         6       P004  2025-03-15    Clinic A  outpatient     NaN  True   

    size  
4  small  
5  small  


**Dropping** removes rows with missing values. Use it with care, because you can lose a lot of data:

In [18]:
claims.dropna(subset=["amount"])

,claim_id,patient_id,date,provider,type,amount,paid,size
0,1,P001,2025-01-04,Clinic A,outpatient,120.50,True,small
1,2,P002,2025-01-09,Hospital B,inpatient,8450.00,True,large
2,3,P001,2025-02-11,Clinic A,outpatient,45.00,False,unpaid
3,4,P003,2025-02-20,Clinic C,pharmacy,15.25,True,small
4,5,P002,2025-03-02,Hospital B,outpatient,310.00,NaN,small
6,7,P003,2025-03-28,Hospital B,inpatient,2200.75,True,large
7,8,P005,2025-04-01,Clinic C,pharmacy,980.00,True,large


**Filling** replaces missing values with something sensible, such as a default or a statistic:

In [19]:
amount_filled = claims["amount"].fillna(claims["amount"].median())
print(amount_filled)

0     120.50
1    8450.00
2      45.00
3      15.25
4     310.00
5     310.00
6    2200.75
7     980.00
Name: amount, dtype: float64


The `paid` column holds `True`, `False`, and missing values, so pandas stores it as `object`. Filling the gaps first, then converting to a real boolean type, makes later comparisons predictable:

In [20]:
claims["paid"] = claims["paid"].fillna(False).astype(bool)
claims[["claim_id", "paid"]]

,claim_id,paid
0,1,True
1,2,True
2,3,False
3,4,True
4,5,False
5,6,True
6,7,True
7,8,True


## 8. Sorting and ranking

In [21]:
claims.sort_values("amount", ascending=False).head(3)

,claim_id,patient_id,date,provider,type,amount,paid,size
1,2,P002,2025-01-09,Hospital B,inpatient,8450.00,True,large
6,7,P003,2025-03-28,Hospital B,inpatient,2200.75,True,large
7,8,P005,2025-04-01,Clinic C,pharmacy,980.00,True,large


In [22]:
claims.sort_values(["provider", "amount"])[["provider", "amount"]]

,provider,amount
2,Clinic A,45.00
0,Clinic A,120.50
5,Clinic A,NaN
3,Clinic C,15.25
7,Clinic C,980.00
4,Hospital B,310.00
6,Hospital B,2200.75
1,Hospital B,8450.00


In [23]:
claims["amount"].rank(ascending=False)

0    5.0
1    1.0
2    6.0
3    7.0
4    4.0
5    NaN
6    2.0
7    3.0
Name: amount, dtype: float64

## 9. Grouping and aggregating

`groupby` splits the data into groups, applies a calculation to each group, and combines the results. It's the pandas version of a SQL `GROUP BY` or a spreadsheet pivot.

In [24]:
claims.groupby("provider")["amount"].sum()

provider
Clinic A        165.50
Clinic C        995.25
Hospital B    10960.75
Name: amount, dtype: float64

Use `.agg` to apply several summaries at once, and name the results:

In [25]:
summary = claims.groupby("type").agg(
    total_claims=("claim_id", "count"),
    total_amount=("amount", "sum"),
    avg_amount=("amount", "mean"),
)
summary

,total_claims,total_amount,avg_amount
type,,,
inpatient,2,10650.75,5325.375
outpatient,4,475.50,158.500
pharmacy,2,995.25,497.625


Groups can use more than one column. Here's the count and total by provider and type:

In [26]:
claims.groupby(["provider", "type"])["amount"].agg(["count", "sum"])

count       sum
provider   type                       
Clinic A   outpatient      2    165.50
Clinic C   pharmacy        2    995.25
Hospital B inpatient       2  10650.75
           outpatient      1    310.00

`value_counts` is a quick way to count how often each value appears:

In [27]:
claims["provider"].value_counts()

provider
Clinic A      3
Hospital B    3
Clinic C      2
Name: count, dtype: int64

`pivot_table` reshapes a grouped result into a grid:

In [28]:
pd.pivot_table(
    claims,
    index="provider",
    columns="type",
    values="amount",
    aggfunc="sum",
    fill_value=0,
)

type,inpatient,outpatient,pharmacy
provider,,,
Clinic A,0.00,165.5,0.00
Clinic C,0.00,0.0,995.25
Hospital B,10650.75,310.0,0.00


## 10. Combining tables

Real projects often keep data in several tables. Suppose we have a table of patients:

In [29]:
patients = pd.DataFrame({
    "patient_id": ["P001", "P002", "P003", "P004", "P005"],
    "plan": ["Gold", "Silver", "Gold", "Bronze", "Silver"],
    "age": [34, 58, 41, 27, 66],
})
patients

,patient_id,plan,age
0,P001,Gold,34
1,P002,Silver,58
2,P003,Gold,41
3,P004,Bronze,27
4,P005,Silver,66


**`merge`** joins two tables on a shared column. This works like a SQL `JOIN`. The `how` argument controls which rows are kept:

- `inner`: only rows present in both tables
- `left`: all rows from the left table, matched where possible

In [30]:
joined = claims.merge(patients, on="patient_id", how="left")
joined[["claim_id", "patient_id", "plan", "age", "amount"]]

,claim_id,patient_id,plan,age,amount
0,1,P001,Gold,34,120.50
1,2,P002,Silver,58,8450.00
2,3,P001,Gold,34,45.00
3,4,P003,Gold,41,15.25
4,5,P002,Silver,58,310.00
5,6,P004,Bronze,27,NaN
6,7,P003,Gold,41,2200.75
7,8,P005,Silver,66,980.00


**`concat`** stacks tables on top of each other, or side by side:

In [31]:
jan = pd.DataFrame({"claim_id": [101], "amount": [50.0]})
feb = pd.DataFrame({"claim_id": [102], "amount": [75.0]})

pd.concat([jan, feb], ignore_index=True)

,claim_id,amount
0,101,50.0
1,102,75.0


## 11. Dates

Once a column holds real dates, you can extract parts of them and do time-based analysis. Convert the text column first:

In [32]:
claims["date"] = pd.to_datetime(claims["date"])

claims["month"] = claims["date"].dt.to_period("M")
claims["weekday"] = claims["date"].dt.day_name()

claims[["date", "month", "weekday"]].head(3)

,date,month,weekday
0,2025-01-04,2025-01,Saturday
1,2025-01-09,2025-01,Thursday
2,2025-02-11,2025-02,Tuesday


In [33]:
claims.groupby("month")["amount"].sum()

month
2025-01    8570.50
2025-02      60.25
2025-03    2510.75
2025-04     980.00
Freq: M, Name: amount, dtype: float64

## 12. Chaining operations

You can write several steps in one expression. Each step returns a new DataFrame, so the chain reads from top to bottom. Wrapping it in parentheses lets you split it across lines:

In [34]:
result = (
    claims
    .dropna(subset=["amount"])
    .query("paid == True")
    .groupby("provider")["amount"]
    .agg(["count", "mean"])
    .round(2)
    .sort_values("mean", ascending=False)
)
result

,count,mean
provider,,
Hospital B,2,5325.38
Clinic C,2,497.62
Clinic A,1,120.50


Chaining is often easier to read than a series of intermediate variables, and it's the style most data code uses.

## 13. Practice exercises

Try each exercise before you look at the solution. Cells marked `# TODO` are for you to fill in.

**Exercise 1.** How many claims are there for each patient? Sort the result from most to fewest claims.

In [46]:
print(claims.groupby("patient_id")["claim_id"].count())
print(claims["patient_id"].value_counts())

patient_id
P001    2
P002    2
P003    2
P004    1
P005    1
Name: claim_id, dtype: int64
patient_id
P001    2
P002    2
P003    2
P004    1
P005    1
Name: count, dtype: int64


<details><summary>Solution</summary>

```python
claims["patient_id"].value_counts()
```

</details>

**Exercise 2.** Find the average amount for paid claims only, by claim type. Ignore rows with a missing amount.

In [51]:
result = (
    claims
    .dropna(subset=["amount"])
    .query("paid == True")
    .groupby("type")["amount"]
    .mean()
)
result

type
inpatient     5325.375
outpatient     120.500
pharmacy       497.625
Name: amount, dtype: float64

<details><summary>Solution</summary>

```python
(
    claims.dropna(subset=["amount"])
    .query("paid == True")
    .groupby("type")["amount"]
    .mean()
)
```

</details>

**Exercise 3.** Add the patient's plan to each claim, then find the total paid amount for each plan.

In [63]:
result = (
    claims
    .merge(patients, on = "patient_id", how = "left")
    .query("paid == True")
    .groupby("plan")["amount"]
    .sum()
)
print(result)

merged = claims.merge(patients, on = "patient_id", how = "left")
merged[merged["paid"]].groupby("plan")["amount"].sum()



plan
Bronze       0.0
Gold      2336.5
Silver    9430.0
Name: amount, dtype: float64


plan
Bronze       0.0
Gold      2336.5
Silver    9430.0
Name: amount, dtype: float64

<details><summary>Solution</summary>

```python
merged = claims.merge(patients, on="patient_id", how="left")
merged[merged["paid"]].groupby("plan")["amount"].sum()
```

</details>

**Exercise 4.** Create a new column `is_large` that is `True` when the amount is over 1,000, and `False` otherwise. Treat missing amounts as `False`.

In [73]:
claims["is_large"] = np.where(
    claims["amount"] > 1000,
      "True", "False")
claims

,claim_id,patient_id,date,provider,type,amount,paid,size,month,weekday,is_large
0,1,P001,2025-01-04,Clinic A,outpatient,120.50,True,small,2025-01,Saturday,False
1,2,P002,2025-01-09,Hospital B,inpatient,8450.00,True,large,2025-01,Thursday,True
2,3,P001,2025-02-11,Clinic A,outpatient,45.00,False,unpaid,2025-02,Tuesday,False
3,4,P003,2025-02-20,Clinic C,pharmacy,15.25,True,small,2025-02,Thursday,False
4,5,P002,2025-03-02,Hospital B,outpatient,310.00,False,small,2025-03,Sunday,False
5,6,P004,2025-03-15,Clinic A,outpatient,NaN,True,small,2025-03,Saturday,False
6,7,P003,2025-03-28,Hospital B,inpatient,2200.75,True,large,2025-03,Friday,True
7,8,P005,2025-04-01,Clinic C,pharmacy,980.00,True,large,2025-04,Tuesday,False


<details><summary>Solution</summary>

```python
claims["is_large"] = (claims["amount"] > 1000).fillna(False)
claims[["amount", "is_large"]]
```

</details>

## Summary

- **Series** are single columns. **DataFrames** are tables made of Series.
- **Load** with `read_csv` and other readers, then **inspect** with `head`, `info`, `describe`, and `isna`.
- Select with `[]` for columns, and `loc` and `iloc` for rows.
- Filter with boolean conditions, combining them with `&`, `|`, and parentheses.
- Create columns by assigning to them, and work on whole columns at once.
- Handle missing values with `isna`, `dropna`, and `fillna`.
- **`groupby`** plus `agg` covers most summary work.
- **`merge`** joins tables, and **`concat`** stacks them.
- Chain steps in parentheses to keep long workflows readable.